# ECON 662 — Problem Set 2
## NFXP Estimation

### Import Libraries 


In [1]:
import time
import numpy as np
import pandas as pd
from scipy.stats import norm        # norm.logpdf — only for OLS sigma check
from scipy.optimize import minimize # outer Nelder-Mead only

# No other packages. Everything else is plain numpy matrix algebra.

###  Load Data


In [2]:
# Columns: i (bus id), t (month 1-100), a (action: 0=maintain, 1=replace),
#          x (mileage state: 1-7), rc (replacement cost, continuous)
df = pd.read_csv("ddc_pset.csv")
df = df.sort_values(["i", "t"]).reset_index(drop=True)

print(f"Loaded {len(df):,} observations for {df['i'].nunique():,} buses.")
print(f"RC range in data: [{df['rc'].min():.4f}, {df['rc'].max():.4f}]")
print(f"Replacement rate: {df['a'].mean():.4f}  ({df['a'].sum():,} replacements)")
print()
df.head(10)

Loaded 100,000 observations for 1,000 buses.
RC range in data: [31.5000, 62.5000]
Replacement rate: 0.1724  (17,241 replacements)



,i,t,a,x,rc
0,1,1,0,4,45.0
1,1,2,0,5,49.0
2,1,3,0,6,54.0
3,1,4,1,7,47.0
4,1,5,0,1,56.0
5,1,6,0,2,52.5
6,1,7,0,3,55.0
7,1,8,0,4,50.5
8,1,9,0,5,52.0
9,1,10,0,6,42.0


### Global Constants

In [3]:
BETA         = 0.95          # discount factor (given in problem)
N_X          = 7             # number of mileage states {1, 2, ..., 7}
N_RC_BINS    = 8             # number of equal-width RC bins (professor uses 8)
GAMMA_EULER  = 0.5772156649  # Euler-Mascheroni constant (mean of Type I EV dist.)

print(f"beta        = {BETA}")
print(f"N_X         = {N_X}  (mileage states)")
print(f"N_RC_BINS   = {N_RC_BINS}  (empirical RC bins)")
print(f"State space = {N_X} x {N_RC_BINS} = {N_X * N_RC_BINS} total states")

beta        = 0.95
N_X         = 7  (mileage states)
N_RC_BINS   = 8  (empirical RC bins)
State space = 7 x 8 = 56 total states


### Step 1: Create Equal-Width RC Bins from the Data

Instead of using the AR(1) theory to build a grid, we simply cut the observed RC range into `N_RC_BINS` equal-width intervals and represent each bin by its midpoint.

> "From now on, rc means the grid mean instead of the actual rc value." 
> Similar to Professor's approach

This fixes the grid once from the data before any estimation begins.

In [4]:
# ── (a) Bin edges: N_RC_BINS+1 evenly spaced points over [RC_min, RC_max] ──
rc_min = df['rc'].min()
rc_max = df['rc'].max()

bin_edges = np.linspace(rc_min, rc_max, N_RC_BINS + 1)   # shape (9,)
print("Bin edges:")
print(np.round(bin_edges, 4))

# ── (b) Bin midpoints: the 'representative RC value' for each bin ──
bin_midpoints = (bin_edges[:-1] + bin_edges[1:]) / 2.0   # shape (8,)
print("\nBin midpoints (rc_grid):")
print(np.round(bin_midpoints, 4))

# ── (c) Assign each observation to its bin ──
# np.searchsorted finds the bin index: bin_edges[j] <= rc < bin_edges[j+1]
# Subtract 1 to make 0-indexed; clip to [0, N_RC_BINS-1] for edge cases
rc_bin_idx = np.searchsorted(bin_edges[1:], df['rc'].values, side='left')
rc_bin_idx = np.clip(rc_bin_idx, 0, N_RC_BINS - 1)       # shape (100000,)

df['rc_bin'] = rc_bin_idx                                 # which bin (0-indexed)
df['rc_mid'] = bin_midpoints[rc_bin_idx]                  # bin representative value

print("\nData after binning (first 10 rows):")
print(df[['i','t','a','x','rc','rc_bin','rc_mid']].head(10))

# ── (d) Observations per bin ──
print("\nObservations per RC bin:")
bin_counts = np.bincount(rc_bin_idx, minlength=N_RC_BINS)
for j in range(N_RC_BINS):
    print(f"  Bin {j}  [{bin_edges[j]:.2f}, {bin_edges[j+1]:.2f}]  "
          f"midpoint={bin_midpoints[j]:.2f}  n={bin_counts[j]:,}")

Bin edges:
[31.5   35.375 39.25  43.125 47.    50.875 54.75  58.625 62.5  ]

Bin midpoints (rc_grid):
[33.4375 37.3125 41.1875 45.0625 48.9375 52.8125 56.6875 60.5625]

Data after binning (first 10 rows):
   i   t  a  x    rc  rc_bin   rc_mid
0  1   1  0  4  45.0       3  45.0625
1  1   2  0  5  49.0       4  48.9375
2  1   3  0  6  54.0       5  52.8125
3  1   4  1  7  47.0       3  45.0625
4  1   5  0  1  56.0       6  56.6875
5  1   6  0  2  52.5       5  52.8125
6  1   7  0  3  55.0       6  56.6875
7  1   8  0  4  50.5       4  48.9375
8  1   9  0  5  52.0       5  52.8125
9  1  10  0  6  42.0       2  41.1875

Observations per RC bin:
  Bin 0  [31.50, 35.38]  midpoint=33.44  n=3,000
  Bin 1  [35.38, 39.25]  midpoint=37.31  n=5,000
  Bin 2  [39.25, 43.12]  midpoint=41.19  n=15,000
  Bin 3  [43.12, 47.00]  midpoint=45.06  n=24,000
  Bin 4  [47.00, 50.88]  midpoint=48.94  n=26,000
  Bin 5  [50.88, 54.75]  midpoint=52.81  n=18,000
  Bin 6  [54.75, 58.62]  midpoint=56.69  n=7,000
  Bi

### Step 2: Build the Empirical Transition Matrix $\Pi$

We count how many times in the data $RC$ moves from bin $j$ to bin $j'$ in consecutive periods:

$$C_{j,j'} = \#\{t : RC_{t-1} \in \text{bin}\, j,\ RC_t \in \text{bin}\, j'\}$$

Then normalize each row to get probabilities:

$$\Pi_{j,j'} = \frac{C_{j,j'}}{\sum_{j''} C_{j,j''}}$$

This matrix $\Pi$ is fixed for the entire estimation, it never changes during optimization.

In [5]:
# (a) Extract consecutive (bin_{t-1}, bin_t) pairs within each bus ──
bin_prev_list, bin_curr_list = [], []

for _, g in df.groupby("i"):
    bins = g['rc_bin'].values
    bin_prev_list.append(bins[:-1])   # bin of RC_{t-1}
    bin_curr_list.append(bins[1:])    # bin of RC_t

bin_prev = np.concatenate(bin_prev_list)  # shape (99000,)
bin_curr = np.concatenate(bin_curr_list)  # shape (99000,)

print(f"Transition pairs extracted: {len(bin_prev):,}")

#  (b) Count matrix C: C[j, j'] = # transitions from bin j to bin j' ──
C = np.zeros((N_RC_BINS, N_RC_BINS), dtype=float)

for j_from, j_to in zip(bin_prev, bin_curr):
    C[j_from, j_to] += 1

print(f"\nCount matrix C  ({N_RC_BINS} x {N_RC_BINS}):")
print("  (C[j, j'] = # times RC moved from bin j to bin j')")
print(C.astype(int))
print(f"\nRow totals (# times RC was in each bin):")
print(C.sum(axis=1).astype(int))

# (c) Normalize rows → transition probability matrix Pi ──
row_totals = C.sum(axis=1, keepdims=True)   # shape (8, 1)
Pi = C / row_totals                          # shape (8, 8)

print(f"\nTransition matrix Pi  ({N_RC_BINS} x {N_RC_BINS}):")
print("  (Pi[j, j'] = P(RC moves to bin j' | currently in bin j))")
print(np.round(Pi, 4))

#  (d) Sanity check: rows must sum to 1 ──
row_sums = Pi.sum(axis=1)
print(f"\nRow sums (all must equal 1.0):")
print(np.round(row_sums, 10))

Transition pairs extracted: 99,000

Count matrix C  (8 x 8):
  (C[j, j'] = # times RC moved from bin j to bin j')
[[   0 1000 2000    0    0    0    0    0]
 [1000 2000 1000 1000    0    0    0    0]
 [2000    0 6000 3000 3000 1000    0    0]
 [   0 2000 2000 7000 9000 2000 1000    0]
 [   0    0 3000 7000 7000 8000 1000    0]
 [   0    0 1000 5000 5000 2000 4000 1000]
 [   0    0    0    0 2000 4000    0 1000]
 [   0    0    0    0    0 1000 1000    0]]

Row totals (# times RC was in each bin):
[ 3000  5000 15000 23000 26000 18000  7000  2000]

Transition matrix Pi  (8 x 8):
  (Pi[j, j'] = P(RC moves to bin j' | currently in bin j))
[[0.     0.3333 0.6667 0.     0.     0.     0.     0.    ]
 [0.2    0.4    0.2    0.2    0.     0.     0.     0.    ]
 [0.1333 0.     0.4    0.2    0.2    0.0667 0.     0.    ]
 [0.     0.087  0.087  0.3043 0.3913 0.087  0.0435 0.    ]
 [0.     0.     0.1154 0.2692 0.2692 0.3077 0.0385 0.    ]
 [0.     0.     0.0556 0.2778 0.2778 0.1111 0.2222 0.0556]
 [0.

We now have an 8 × 8 matrix $\Pi$ built entirely from data

### Step 3:  OLS for AR(1) Parameters

We estimate $(\rho_0, \rho_1, \sigma_\rho)$ once via OLS before the optimizer runs.
The AR(1) likelihood is separable from the choice likelihood; therefore, we can maximise each part independently.

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^\top \mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$$

where $\mathbf{X} = [\mathbf{1} \mid RC_{t-1}]$ and $\mathbf{y} = RC_t$.

In [6]:
# (a) Build (RC_{t-1}, RC_t) pairs using actual (continuous) RC values 
rc_prev_list, rc_curr_list = [], []
for _, g in df.groupby("i"):
    rc_vals = g['rc'].values
    rc_prev_list.append(rc_vals[:-1])
    rc_curr_list.append(rc_vals[1:])

rc_prev_vec = np.concatenate(rc_prev_list)   # shape (99000,)
rc_curr_vec = np.concatenate(rc_curr_list)   # shape (99000,)

# (b) Design matrix X = [1, RC_{t-1}], shape (99000, 2) 
X = np.column_stack([np.ones(len(rc_prev_vec)), rc_prev_vec])

print(f"Design matrix X shape: {X.shape}")
print("First 5 rows of X:")
print(X[:5])

# (c) OLS normal equations: β = (X'X)^{-1} X'y 
XtX = X.T @ X          # (2, 2)
Xty = X.T @ rc_curr_vec  # (2,)

print(f"\nX'X =\n{XtX}")
print(f"\nX'y = {Xty}")

beta_ols = np.linalg.solve(XtX, Xty)
rho0_ols, rho1_ols = beta_ols

resid_ols    = rc_curr_vec - X @ beta_ols
sigma_rho_ols = resid_ols.std(ddof=1)

print("\n--- OLS AR(1) estimates (fixed for entire estimation) ---")
print(f"  rho0      = {rho0_ols:.4f}")
print(f"  rho1      = {rho1_ols:.4f}")
print(f"  sigma_rho = {sigma_rho_ols:.4f}")
#print("\nThese match Table 1 in the professor's solution: rho0=17.8269, rho1=0.6249, sigma=4.606")

Design matrix X shape: (99000, 2)
First 5 rows of X:
[[ 1. 45.]
 [ 1. 49.]
 [ 1. 54.]
 [ 1. 47.]
 [ 1. 56.]]

X'X =
[[9.9000000e+04 4.7025000e+06]
 [4.7025000e+06 2.2682125e+08]]

X'y = [4.7035000e+06 2.2557375e+08]

--- OLS AR(1) estimates (fixed for entire estimation) ---
  rho0      = 17.8269
  rho1      = 0.6249
  sigma_rho = 4.6060


### Step 4: State-Level Aggregation

From the professor notes: "Notice that all the estimations are done at state level"

There are $7 \times 8 = 56$ states. For each state $(x, j)$ we record:
- $N_1(x, j)$ = number of observations where action = 1 (replace)
- $N_0(x, j)$ = number of observations where action = 0 (maintain)

The likelihood then sums over states (56 terms), not observations (100,000 terms).
#Same mathematical result, much faster computation.

In [7]:
#  Count N_0 and N_1 for each state (x, rc_bin) 
# State matrix shape: (N_X, N_RC_BINS)
# x is 1-indexed in data, rc_bin is 0-indexed

N1 = np.zeros((N_X, N_RC_BINS))   # N1[xi, j] = # replace decisions in state (xi+1, j)
N0 = np.zeros((N_X, N_RC_BINS))   # N0[xi, j] = # maintain decisions in state (xi+1, j)

x_data   = df['x'].values.astype(int)        # 1-indexed
bin_data = df['rc_bin'].values.astype(int)   # 0-indexed
a_data   = df['a'].values.astype(int)

for xi_obs, j_obs, a_obs in zip(x_data, bin_data, a_data):
    if a_obs == 1:
        N1[xi_obs - 1, j_obs] += 1   # subtract 1 for 0-indexing
    else:
        N0[xi_obs - 1, j_obs] += 1

print(f"N1 shape: {N1.shape}  (rows=mileage x, cols=RC bin)")
print(f"\nN1  — # REPLACE decisions per state (x, rc_bin):")
print("        ", [f"bin{j}" for j in range(N_RC_BINS)])
for xi in range(N_X):
    print(f"  x={xi+1}:  {N1[xi].astype(int)}")

print(f"\nN0  — # MAINTAIN decisions per state (x, rc_bin):")
for xi in range(N_X):
    print(f"  x={xi+1}:  {N0[xi].astype(int)}")

print(f"\nTotal observations: N0+N1 = {(N0+N1).sum():.0f}  (should be 100,000)")

#  Empirical replacement rate per state 
N_total = N0 + N1
P_emp = np.where(N_total > 0, N1 / N_total, np.nan)  # empirical CCP at each state
print("\nEmpirical P(replace) per state (raw frequency):")
print("        ", [f"bin{j}" for j in range(N_RC_BINS)])
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(P_emp[xi], 3)}")

N1 shape: (7, 8)  (rows=mileage x, cols=RC bin)

N1  — # REPLACE decisions per state (x, rc_bin):
         ['bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5', 'bin6', 'bin7']
  x=1:  [20 23 29 22 15  7  0  0]
  x=2:  [42 67 94 89 65 20  4  2]
  x=3:  [172 158 339 302 220 107  15   3]
  x=4:  [187 237 615 693 533 225  63   9]
  x=5:  [ 200  274  788 1044  876  447  143   20]
  x=6:  [  91  221  642 1039  844  647  144   22]
  x=7:  [ 185  248  687 1369 1433 1094  323   83]

N0  — # MAINTAIN decisions per state (x, rc_bin):
  x=1:  [ 682 1164 3091 4075 4092 2800 1023  237]
  x=2:  [ 499  942 3007 4043 4373 2616 1045  236]
  x=3:  [ 477  753 2183 3728 4315 2615 1109  270]
  x=4:  [ 239  454 1604 3365 3619 2395  880  357]
  x=5:  [ 122  242 1045 2011 2604 1940  925  223]
  x=6:  [  45  130  495 1166 1443 1424  497  221]
  x=7:  [  39   87  381 1054 1568 1663  829  317]

Total observations: N0+N1 = 100000  (should be 100,000)

Empirical P(replace) per state (raw frequency):
         ['bin0', '

### Bellman Equation 

$$v_0(x,j) = -\theta_1 x + \beta \underbrace{\sum_{j'} \Pi_{j,j'} EV(\min(x{+}1,7),\ j')}_{\text{row } j \text{ of } EV \cdot \Pi^\top}$$

$$v_1(x,j) = -\theta_2\, rc_j + \beta \underbrace{\sum_{j'} \Pi_{j,j'} EV(1,\ j')}_{\text{row } j \text{ of } EV \cdot \Pi^\top, \text{ first row only}}$$

$$EV(x,j) = \log(e^{v_0} + e^{v_1}) + \gamma_E$$


In [8]:
def solve_bellman(theta1, theta2, rc_grid, Pi, tol=1e-8, maxiter=2000):
    """
    Value function iteration.
    Pi is now 8x8 and fixed; rc_grid is the 8 bin midpoints.

    Returns
    -------
    EV      : ndarray (N_X, N_RC_BINS)  — integrated value function
    v0_grid : ndarray (N_X, N_RC_BINS)  — value of maintaining
    v1_grid : ndarray (N_X, N_RC_BINS)  — value of replacing
    """
    N_rc = len(rc_grid)
    EV   = np.zeros((N_X, N_rc))   # start from zero

    # Pre-compute objects that never change across iterations
    x_idx    = np.arange(N_X)                    # [0, 1, 2, 3, 4, 5, 6]
    x_next_0 = np.minimum(x_idx + 1, N_X - 1)   # maintain: x+1, capped at 6 (state 7)
    x_next_1 = 0                                  # replace: always reset to state 1

    # Flow utility of maintenance: -theta1 * x, shape (N_X, 1)
    u0_base = (-theta1 * (x_idx + 1)).reshape(N_X, 1)

    # Flow utility of replacement: -theta2 * rc_grid[j], repeated across mileage states
    u1 = np.broadcast_to((-theta2 * rc_grid).reshape(1, N_rc), (N_X, N_rc))

    for iteration in range(maxiter):

        # ── Continuation values: one matrix multiply 
        # EV_cont[xi, j] = sum_{j'} Pi[j, j'] * EV[xi, j']
        #                 = (EV @ Pi.T)[xi, j]
        # Shape: (N_X, N_rc) = (7, 8) @ (8, 8) transposed... wait:
        # EV is (7, 8), Pi.T is (8, 8), so EV @ Pi.T is (7, 8)  
        EV_cont = EV @ Pi.T                              # (7, 8)

        # ── Choice-specific value functions 
        v0 = u0_base + BETA * EV_cont[x_next_0, :]      # (7, 8)
        v1 = u1 + np.broadcast_to(EV_cont[x_next_1, :], (N_X, N_rc)) * BETA

        # ── Log-sum-exp (numerically stable) 
        vmax   = np.maximum(v0, v1)
        EV_new = np.log(np.exp(v0 - vmax) + np.exp(v1 - vmax)) + vmax + GAMMA_EULER

        diff = np.max(np.abs(EV_new - EV))
        EV   = EV_new

        if diff < tol:
            break

    return EV, v0, v1

#### Inspect the Bellman output at initial guess $\theta_1 = \theta_2 = 1$

In [9]:
rc_grid = bin_midpoints   # the 8 bin midpoints — our RC grid

EV_init, v0_init, v1_init = solve_bellman(1.0, 1.0, rc_grid, Pi)

print(f"rc_grid (bin midpoints): {np.round(rc_grid, 2)}")
print(f"\nEV shape: {EV_init.shape}  (7 mileage states × 8 RC bins)")

print("\n── EV (integrated value function) ──")
print(f"{'':6}", "  ".join(f"bin{j}" for j in range(N_RC_BINS)))
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(EV_init[xi], 2)}")

print("\n── v0_grid (value of MAINTAINING) ──")
print(f"{'':6}", "  ".join(f"bin{j}" for j in range(N_RC_BINS)))
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(v0_init[xi], 2)}")

print("\n── v1_grid (value of REPLACING) ──")
print(f"{'':6}", "  ".join(f"bin{j}" for j in range(N_RC_BINS)))
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(v1_init[xi], 2)}")

print("\n── v1 - v0  (positive => prefer replace) ──")
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(v1_init[xi] - v0_init[xi], 3)}")

rc_grid (bin midpoints): [33.44 37.31 41.19 45.06 48.94 52.81 56.69 60.56]

EV shape: (7, 8)  (7 mileage states × 8 RC bins)

── EV (integrated value function) ──
       bin0  bin1  bin2  bin3  bin4  bin5  bin6  bin7
  x=1:  [-109.12 -109.12 -109.12 -109.12 -109.12 -109.12 -109.12 -109.12]
  x=2:  [-114.42 -114.42 -114.42 -114.42 -114.42 -114.42 -114.42 -114.42]
  x=3:  [-118.94 -118.94 -118.94 -118.94 -118.94 -118.94 -118.94 -118.94]
  x=4:  [-122.65 -122.65 -122.65 -122.65 -122.65 -122.65 -122.65 -122.65]
  x=5:  [-125.51 -125.51 -125.51 -125.51 -125.51 -125.51 -125.51 -125.51]
  x=6:  [-127.46 -127.46 -127.46 -127.46 -127.46 -127.46 -127.46 -127.46]
  x=7:  [-128.46 -128.46 -128.46 -128.46 -128.46 -128.46 -128.46 -128.46]

── v0_grid (value of MAINTAINING) ──
       bin0  bin1  bin2  bin3  bin4  bin5  bin6  bin7
  x=1:  [-109.7 -109.7 -109.7 -109.7 -109.7 -109.7 -109.7 -109.7]
  x=2:  [-115. -115. -115. -115. -115. -115. -115. -115.]
  x=3:  [-119.52 -119.52 -119.52 -119.52 -119.52 

### Step 6: Choice Probabilities

Here, we already replaced every observed RC with its bin midpoint.
So each observation maps to exactly one grid index, just a direct array lookup:

$$P(\text{replace} \mid x_i, \text{bin}_i) = \frac{1}{1 + \exp(v_0[x_i,\, \text{bin}_i] - v_1[x_i,\, \text{bin}_i])}$$

Vectorised over all 100,000 observations in one line, or, equivalently, over all 56 states.

In [10]:
def compute_choice_probs_grid(v0_grid, v1_grid):
    """
    Compute P(replace | x, rc_bin) for every (x, rc_bin) state.

    No interpolation needed — v0 and v1 are already on the exact grid.
    Returns the full (N_X, N_RC_BINS) matrix of replacement probabilities.
    """
    # P(replace | x, j) = 1 / (1 + exp(v0 - v1))
    return 1.0 / (1.0 + np.exp(v0_grid - v1_grid))   # (7, 8)


# ── Compute at initial guess ──
P_pred_init = compute_choice_probs_grid(v0_init, v1_init)   # (7, 8)

print("Predicted P(replace | x, rc_bin)  at theta1=1, theta2=1:")
print(f"{'':6}", "  ".join(f"bin{j}" for j in range(N_RC_BINS)))
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(P_pred_init[xi], 3)}")

print("\nEmpirical P(replace | x, rc_bin)  (from data frequency):")
for xi in range(N_X):
    print(f"  x={xi+1}:  {np.round(P_emp[xi], 3)}")

Predicted P(replace | x, rc_bin)  at theta1=1, theta2=1:
       bin0  bin1  bin2  bin3  bin4  bin5  bin6  bin7
  x=1:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=2:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=3:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=4:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=5:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=6:  [0. 0. 0. 0. 0. 0. 0. 0.]
  x=7:  [0. 0. 0. 0. 0. 0. 0. 0.]

Empirical P(replace | x, rc_bin)  (from data frequency):
  x=1:  [0.028 0.019 0.009 0.005 0.004 0.002 0.    0.   ]
  x=2:  [0.078 0.066 0.03  0.022 0.015 0.008 0.004 0.008]
  x=3:  [0.265 0.173 0.134 0.075 0.049 0.039 0.013 0.011]
  x=4:  [0.439 0.343 0.277 0.171 0.128 0.086 0.067 0.025]
  x=5:  [0.621 0.531 0.43  0.342 0.252 0.187 0.134 0.082]
  x=6:  [0.669 0.63  0.565 0.471 0.369 0.312 0.225 0.091]
  x=7:  [0.826 0.74  0.643 0.565 0.478 0.397 0.28  0.208]


### Step 7: Log-Likelihood at the State Level

The likelihood collapses to a state-level sum:

$$\ell(\theta_1, \theta_2) = \sum_{s=1}^{S} \Bigl[
  \hat{P}(a{=}1 \mid x_s, rc_s,\, \theta)\cdot N_1(x_s, rc_s)
  + \hat{P}(a{=}0 \mid x_s, rc_s,\, \theta)\cdot N_0(x_s, rc_s)
\Bigr]$$

where $S = 56$ states, and $N_1, N_0$ are the count matrices.


In [11]:
def neg_log_likelihood(params):
    """
    Negative log-likelihood: choice component only.
    AR(1) params are pre-estimated and Pi is pre-built — neither appears here.

    params : [theta1, theta2]
    """
    theta1, theta2 = params

    # Guard rails: cost parameters must be positive
    if theta1 <= 0 or theta2 <= 0:
        return 1e10

    #  Step A: Bellman fixed point (Pi is fixed from data) 
    _, v0_grid, v1_grid = solve_bellman(theta1, theta2, rc_grid, Pi)

    #  Step B: Predicted P(replace) at every state 
    P_rep = compute_choice_probs_grid(v0_grid, v1_grid)   # (N_X, N_RC_BINS)
    P_mnt = 1.0 - P_rep                                   # P(maintain)

    #  Step C: State-level log-likelihood 
    # Sum over all 56 states: N1[x,j] * log P(rep) + N0[x,j] * log P(mnt)
    eps = 1e-12   # prevent log(0)
    ll = np.sum(N1 * np.log(P_rep + eps) + N0 * np.log(P_mnt + eps))

    return -ll   # return negative LL (scipy minimises)


#  Check at initial guess 
params_init = np.array([1.0, 1.0])
nll_init = neg_log_likelihood(params_init)
print(f"Negative log-likelihood at theta1=1, theta2=1:  {nll_init:.4f}")
print(f"(Log-likelihood = {-nll_init:.4f})")

Negative log-likelihood at theta1=1, theta2=1:  386763.3267
(Log-likelihood = -386763.3267)


### Step 8: Optimisation

The outer loop now searches over only $\theta_1$ and $\theta_2$.
$\Pi$, the AR(1) parameters, and the RC grid are all fixed.

In [12]:
print("--- Starting NFXP Optimisation (empirical Pi, 2 params) ---")
print(f"  Initial theta1 = {params_init[0]}")
print(f"  Initial theta2 = {params_init[1]}")
print(f"  Initial neg-LL = {neg_log_likelihood(params_init):.4f}\n")

t0_theta = time.perf_counter()
result = minimize(
    neg_log_likelihood,
    params_init,
    method="Nelder-Mead",
    options=dict(
        maxiter = 5000,
        xatol   = 1e-8,
        fatol   = 1e-8,
        disp    = True,
    )
)
theta_runtime_sec = time.perf_counter() - t0_theta
theta_runtime_ms = 1000.0 * theta_runtime_sec
running_time_sec = theta_runtime_sec
running_time_ms = theta_runtime_ms
print(f"\nComparable running time: {running_time_sec:.4f} seconds ({running_time_ms:.2f} ms)")

--- Starting NFXP Optimisation (empirical Pi, 2 params) ---
  Initial theta1 = 1.0
  Initial theta2 = 1.0
  Initial neg-LL = 386763.3267



Optimization terminated successfully.
         Current function value: 33977.621162
         Iterations: 76
         Function evaluations: 150

Comparable running time: 0.8161 seconds (816.07 ms)


### Results

In [13]:
theta1_hat, theta2_hat = result.x

print("=" * 55)
print("      NFXP ESTIMATION RESULTS")
print("=" * 55)
print(f"  theta1    (mileage cost coeff.)    = {theta1_hat:.4f}")
print(f"  theta2    (replacement cost coeff) = {theta2_hat:.4f}")
print(f"  ---  pre-estimated  ---")
print(f"  rho0      (AR(1) intercept)        = {rho0_ols:.4f}")
print(f"  rho1      (AR(1) persistence)      = {rho1_ols:.4f}")
print(f"  sigma_rho (AR(1) std dev)          = {sigma_rho_ols:.4f}")
print("-" * 55)
print(f"  Log-likelihood (choice only)       = {-result.fun:.4f}")
print(f"  Converged: {result.success}   |   Iterations: {result.nit}")
print(f"  Comparable running time            = {running_time_sec:.4f} sec ({running_time_ms:.2f} ms)")
print("=" * 55)

#print("\nComparison with professor Jingyuan:")
#print(f"  Professor:  theta1 = 0.4118   theta2 = 0.1567")
#print(f"  This code:  theta1 = {theta1_hat:.4f}   theta2 = {theta2_hat:.4f}")

      NFXP ESTIMATION RESULTS
  theta1    (mileage cost coeff.)    = 0.4088
  theta2    (replacement cost coeff) = 0.1563
  ---  pre-estimated  ---
  rho0      (AR(1) intercept)        = 17.8269
  rho1      (AR(1) persistence)      = 0.6249
  sigma_rho (AR(1) std dev)          = 4.6060
-------------------------------------------------------
  Log-likelihood (choice only)       = -33977.6212
  Converged: True   |   Iterations: 76
  Comparable running time            = 0.8161 sec (816.07 ms)


## Summary: What We Did (and Why)

### Algorithm flow

```
ONCE (before optimizer):
  1. Cut observed RC into 8 equal-width bins → rc_grid (bin midpoints)
  2. Count (bin_{t-1} → bin_t) transitions → count matrix C  →  Pi = C / row_sums
  3. OLS on (RC_{t-1}, RC_t) pairs → rho0, rho1, sigma_rho  (fixed)
  4. Count N1[x,j] and N0[x,j] for all 56 states

OUTER LOOP (Nelder-Mead over theta1, theta2 only):
  ├── INNER LOOP: Bellman iteration using fixed Pi and rc_grid
  │     EV_cont = EV @ Pi.T          ← matrix multiply
  │     v0 = u0 + β * EV_cont[x+1, :]
  │     v1 = u1 + β * EV_cont[1,   :]
  │     EV = log(exp(v0) + exp(v1)) + γ_E
  │     repeat until max|ΔEV| < 1e-8
  │
  ├── P(replace|x,j) = 1 / (1 + exp(v0 - v1))    ← no interpolation
  └── ℓ = Σ_{x,j} [N1[x,j] log P_rep + N0[x,j] log P_mnt]
```

### Why no Tauchen?

Tauchen is needed when your grid changes with parameters —
i.e., when the AR(1) parameters are inside the optimizer.
Here, $\Pi$ is built from the **data directly** and is never recomputed.
The grid is fixed, so no Tauchen formula is needed.

### Tradeoff
| | Empirical (this notebook) | Tauchen |
|---|---|---|
| Grid built from | Data (bins) | AR(1) theory |
| $\Pi$ recomputed? | Never | Every optimizer call |
| Params in optimizer | 2 | 5 |
| Empty-cell problem? | Yes (rare states) | No |
| Speed | Fast | Slower |